In [ ]:
# Import Functions
from functions import JiraAPI2 as Jira
import pandas as pd
from time import sleep
import csv
from datetime import date, timedelta, datetime
from functions import dbCall as dbcc

In [ ]:
# Check API Connection
Jira.fCheckAPIConnection()

------------
Previous Date Pull

In [ ]:
# Full download of Jira data for today's date:
effectivedate = date.today() - timedelta(days=1)
effectivedate = effectivedate.strftime('%Y-%m-%d')
dailytickets, comments, changelog, SLA = Jira.fJiraExport(effectivedate=effectivedate)

In [ ]:
# Uploads the results to the database | Must be connected to VPN:
Jira.fUploadJiraDataToDB_Tickets(dailytickets=dailytickets)
Jira.fUploadJiraDataToDB_Comments(comments=comments)
Jira.fUploadJiraDataToDB_ChangeLog(changelog=changelog)
Jira.fUploadJiraDataToDB_SLA(sla=SLA)
Jira.fRunStoredProcedures()

______
SQL Guided Pull

In [ ]:
sql = """SELECT dateadd(dd,1,max(cast(Updated as date))) as STARTDATE, cast(getdate()-1 as date) as ENDDATE FROM IceAutomation_AZR.jira.TicketTable"""
effdates = dbcc.SQL_Call_pandas(sql=sql)
sd = str(effdates.loc[0, 'STARTDATE'])
ed = str(effdates.loc[0, 'ENDDATE'])
date_list = pd.date_range(start=sd, end=ed)
for i in date_list:
    effectivedate = i.strftime('%Y-%m-%d')
    print(effectivedate)
    Jira.fFullRun(effectivedate=effectivedate)   
    sleep(5)

-----------
Date Range Pull

In [ ]:
date_list = pd.date_range(start='2025-12-12', end='2025-12-14')
for i in date_list:
    effectivedate = i.strftime('%Y-%m-%d')
    print(effectivedate)
    Jira.fFullRun(effectivedate=effectivedate)   
    sleep(5)